# MLflow training summary

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns

from gen_cats.evaluation.experiment_artifacts import load_mlflow_runs, training_summary_by_model
from gen_cats.evaluation.report_analysis import ensure_plots_dir, write_snippet, write_stats_csv

sns.set_theme(style="whitegrid", context="talk")
PLOTS = ensure_plots_dir("results")
ml = load_mlflow_runs()
summary = training_summary_by_model(ml)
summary

,model_type,model_label,runs_total,runs_finished,runs_failed,mean_final_epoch,median_final_epoch,early_stop_rate,mean_best_metric,median_duration_min,mean_duration_min,median_sec_per_epoch,mean_sec_per_epoch
5,vqvae,VQ-VAE-1,57,30,27,150.766667,151.0,1.0,0.088873,46.718150,45.599601,18.354094,17.921823
6,wgan_gp,WGAN-GP,41,33,8,41.121212,40.0,1.0,-91.181111,122.205617,142.328213,187.189927,202.405144
0,beta_vae,$\beta$-VAE,38,24,14,50.708333,42.5,1.0,0.220217,19.232267,21.979227,26.978759,25.724669
3,sn_gan,SN-GAN,26,24,2,46.541667,43.0,1.0,-0.952582,38.307025,41.673269,53.444675,53.688364
1,ddim,DDIM,21,17,4,50.411765,46.0,1.0,0.034802,144.712433,194.451882,130.896931,232.983541
4,tiny_ldm,Tiny LDM,20,19,1,65.368421,50.0,1.0,0.384233,10.518817,12.051708,9.847750,12.151991
2,pixelcnn,PixelCNN,7,6,1,146.333333,161.5,1.0,2.884026,83.384358,65.765572,30.368488,24.876323


In [2]:
plot_df = summary.sort_values("median_sec_per_epoch", ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
axes[0].barh(plot_df["model_label"], plot_df["runs_finished"], color="#55A868", label="finished")
axes[0].barh(
    plot_df["model_label"],
    plot_df["runs_failed"],
    left=plot_df["runs_finished"],
    color="#C44E52",
    label="failed",
)
axes[0].set_title("Run outcomes")
axes[0].legend(loc="lower right", fontsize=9)
axes[1].barh(plot_df["model_label"], plot_df["median_final_epoch"])
axes[1].set_title("Median epochs")
axes[2].barh(plot_df["model_label"], plot_df["median_sec_per_epoch"] / 60.0)
axes[2].set_title("Median min / epoch")
axes[2].set_xlabel("minutes")
fig.tight_layout()
out = PLOTS / "mlflow_run_summary.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.close(fig)
out

PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/notebooks/plots/results/mlflow_run_summary.png')

In [3]:
latex_lines = [
    r"\begin{table}[H]",
    r"    \centering",
    r"    \caption{MLflow run outcomes and training duration (finished runs on Apple MPS).}",
    r"    \label{tab:mlflow_summary}",
    r"    \small",
    r"    \resizebox{\linewidth}{!}{%",
    r"    \begin{tabular}{lrrrrrr}",
    r"        \toprule",
    r"        Model & OK & Fail & Med.\ ep. & Med.\ time [min] & Med.\ s/ep & ES [\%] \\",
    r"        \midrule",
]
for _, row in summary.sort_values("model_label").iterrows():
    es = row["early_stop_rate"]
    es_txt = f"{100 * es:.0f}" if es == es else "---"
    med_ep = row["median_final_epoch"]
    med_ep_txt = f"{med_ep:.0f}" if med_ep == med_ep else "---"
    med_time = row["median_duration_min"]
    med_time_txt = f"{med_time:.0f}" if med_time == med_time else "---"
    med_spe = row["median_sec_per_epoch"]
    med_spe_txt = f"{med_spe:.0f}" if med_spe == med_spe else "---"
    label = row["model_label"]
    finished = int(row["runs_finished"])
    failed = int(row["runs_failed"])
    latex_lines.append(
        f"        {label} & {finished} & {failed} & {med_ep_txt} & "
        f"{med_time_txt} & {med_spe_txt} & {es_txt} \\\\"
    )
latex_lines += [
    r"        \bottomrule",
    r"    \end{tabular}%",
    r"    }",
    r"\end{table}",
]
write_snippet("mlflow_summary.tex", "\n".join(latex_lines))
write_stats_csv(summary, "mlflow_summary.csv")
ml.groupby(["model_type", "status"]).size().unstack(fill_value=0)

status,FAILED,FINISHED
model_type,,
beta_vae,14,24
ddim,4,17
pixelcnn,1,6
sn_gan,3,24
tiny_ldm,1,19
vqvae,27,30
wgan_gp,10,37
